# Chat with Text (RAG)
This notebook implements a Retrieval-Augmented Generation (RAG) pipeline to answer questions about the State of the Union address using a local embedding model and Llama 3 via OpenRouter.

In [1]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI 

/home/miad/projects/RAG/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Setup Environment
Define API keys for OpenRouter.

In [2]:
# SETUP: OpenRouter
# Get key from: https://openrouter.ai/keys
os.environ["OPENAI_API_KEY"] = "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" # Your OpenRouter Key
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

### 1. Load Documents
Read the text file into memory.

In [3]:
# 1. Load Data
loader = TextLoader("./state_of_the_union.txt")
docs = loader.load()

In [4]:
docs

[Document(metadata={'source': './state_of_the_union.txt'}, page_content='My fellow Americans,\n\nTonight, I want to talk about the future of our nation. We face many challenges, but also incredible opportunities.\n\nOne of the greatest opportunities lies in the field of technology. We are investing in American innovation. We are building the next generation of semiconductor chips right here in Ohio. We are ensuring that artificial intelligence is developed safely and responsibly, to help us cure diseases and combat climate change, not to undermine our democracy.\n\nWe are also expanding high-speed internet to every corner of this country, because in the 21st century, high-speed internet is not a luxury, it is a necessity.\n\nLet us move forward together, embracing the future with confidence and hope.\n\nGod bless you all, and may God protect our troops.\n')]

### 2. Split Text
Break the document into chunks for processing.

In [5]:
# 2. Split Data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

In [6]:
print(splits)

[Document(metadata={'source': './state_of_the_union.txt'}, page_content='My fellow Americans,\n\nTonight, I want to talk about the future of our nation. We face many challenges, but also incredible opportunities.\n\nOne of the greatest opportunities lies in the field of technology. We are investing in American innovation. We are building the next generation of semiconductor chips right here in Ohio. We are ensuring that artificial intelligence is developed safely and responsibly, to help us cure diseases and combat climate change, not to undermine our democracy.\n\nWe are also expanding high-speed internet to every corner of this country, because in the 21st century, high-speed internet is not a luxury, it is a necessity.\n\nLet us move forward together, embracing the future with confidence and hope.\n\nGod bless you all, and may God protect our troops.')]


### 3. Vector Store & Embeddings
Create local embeddings and index them in ChromaDB.

In [7]:
print("Generating embeddings...")
# embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)
retriever = vectorstore.as_retriever()

Generating embeddings...


### 4. Initialize LLM
Connect to the Llama 3 model via OpenRouter.

In [8]:
llm = ChatOpenAI(
    model="x-ai/grok-4.1-fast",
    # model="meta-llama/llama-3-8b-instruct",
    temperature=0.2
)

### 5. Build RAG Chain
Combine the retriever, prompt, and LLM into a query pipeline.

In [9]:
# 5. The RAG Chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### 6. Execute Query
Run the chain to answer a question.

In [10]:
# 6. Ask
print("Asking LLM...")
response = rag_chain.invoke("What did the president say about technology?")
print(response)

Asking LLM...
The president described technology as one of the greatest opportunities for the nation. He highlighted:

- Investing in American innovation.
- Building the next generation of semiconductor chips in Ohio.
- Ensuring artificial intelligence is developed safely and responsibly to cure diseases and combat climate change, rather than undermine democracy.
- Expanding high-speed internet to every corner of the country, calling it a necessity in the 21st century.
